In [ ]:
print("\n" + "="*80)
print("ANALYSIS SUMMARY")
print("="*80)
print(f"\nPeaks filtered to 20kb threshold:")
print(f"  Male: {len(peak_annotation_Male_20000)}")
print(f"  Female: {len(peak_annotation_Female_20000)}")
print(f"  Combined: {len(all_peaks)}")

print(f"\nUnique genes targeted:")
print(f"  Male: {len(male_genes)}")
print(f"  Female: {len(female_genes)}")
print(f"  Combined: {len(all_genes)}")
print(f"  Shared: {len(male_genes & female_genes)}")

print(f"\nSex-biased peak overlaps:")
print(f"  Combined peaks overlapping Male > Female: {len(overlap_with_male_biased_df)} ({100*len(overlap_with_male_biased_df)/len(all_peaks):.1f}%)")
print(f"  Combined peaks overlapping Female > Male: {len(overlap_with_female_biased_df)} ({100*len(overlap_with_female_biased_df)/len(all_peaks):.1f}%)")

print(f"\nOutput files saved:")
print(f"  - Male_Egr1CC_peaks_20kbThreshhold.txt")
print(f"  - Female_Egr1CC_peaks_20kbThreshhold.txt")
print(f"  - Male_unique_genes_20kb.txt")
print(f"  - Female_unique_genes_20kb.txt")
print(f"  - All_unique_genes_20kb.txt")
print(f"  - Combined_20kb_peaks_vs_MaleEgr1vsFemaleEgr1_overlaps.txt")
print(f"  - Combined_20kb_peaks_vs_FemaleEgr1vsMaleEgr1_overlaps.txt")
print("="*80)

## 8. Summary Statistics

In [ ]:
# Intersect combined peaks with Female > Male peaks
overlap_with_female_biased = combined_bed.intersect(female_vs_male_bed, wo=True)
overlap_with_female_biased_df = overlap_with_female_biased.to_dataframe()
overlap_with_female_biased_df.columns = ['combined_chr', 'combined_start', 'combined_end', 
                                          'female_chr', 'female_start', 'female_end', 'overlap_bp']

print(f"\nCombined 20kb peaks overlapping Female > Male: {len(overlap_with_female_biased_df)}")
print(f"  {100*len(overlap_with_female_biased_df)/len(all_peaks):.1f}% of combined peaks")

overlap_with_female_biased_df.to_csv("Combined_20kb_peaks_vs_FemaleEgr1vsMaleEgr1_overlaps.txt", 
                                      sep='\t', index=False, header=True)
print("Saved: Combined_20kb_peaks_vs_FemaleEgr1vsMaleEgr1_overlaps.txt")

## 7. Intersect combined 20kb peaks with Female-biased peaks

In [ ]:
# Intersect combined peaks with Male > Female peaks
overlap_with_male_biased = combined_bed.intersect(male_vs_female_bed, wo=True)
overlap_with_male_biased_df = overlap_with_male_biased.to_dataframe()
overlap_with_male_biased_df.columns = ['combined_chr', 'combined_start', 'combined_end', 
                                        'male_chr', 'male_start', 'male_end', 'overlap_bp']

print(f"\nCombined 20kb peaks overlapping Male > Female: {len(overlap_with_male_biased_df)}")
print(f"  {100*len(overlap_with_male_biased_df)/len(all_peaks):.1f}% of combined peaks")

overlap_with_male_biased_df.to_csv("Combined_20kb_peaks_vs_MaleEgr1vsFemaleEgr1_overlaps.txt", 
                                    sep='\t', index=False, header=True)
print("Saved: Combined_20kb_peaks_vs_MaleEgr1vsFemaleEgr1_overlaps.txt")

## 6. Intersect combined 20kb peaks with Male-biased peaks

In [ ]:
# Define peak columns for sex-biased files
peak_columns = ['Chr', 'Start', 'End', 'Center', 'Experiment Insertions', 'Background insertions', 
                'Reference Insertions', 'pvalue Reference', 'pvalue Background', 'Fraction Experiment', 
                'TPH Experiment', 'Fraction background', 'TPH background', 'TPH background subtracted', 
                'pvalue_adj Reference']

# Load Male vs Female peaks
male_vs_female = pd.read_csv("Egr1CC_peak_MaleEgr1_VS_FemaleEgr1_MACC2_YchromFiltered_window300_bg005_p05_TTAAp001_052125.bed", 
                              sep='\t', header=None)
male_vs_female.columns = peak_columns
print(f"Male Egr1 > Female Egr1: {len(male_vs_female)} peaks")

# Load Female vs Male peaks
female_vs_male = pd.read_csv("Egr1CC_peak_FemaleEgr1_VS_MaleEgr1_MACC2_YchromFiltered_window300_bg005_p05_TTAAp001_052125.bed", 
                              sep='\t', header=None)
female_vs_male.columns = peak_columns
print(f"Female Egr1 > Male Egr1: {len(female_vs_male)} peaks")

# Create bedtools objects
male_vs_female_bed = BedTool.from_dataframe(male_vs_female[['Chr', 'Start', 'End']])
female_vs_male_bed = BedTool.from_dataframe(female_vs_male[['Chr', 'Start', 'End']])

## 5. Load sex-biased peak files and intersect

In [ ]:
# Combine all 20kb-filtered peaks
all_peaks = pd.concat([peak_annotation_Male_20000, peak_annotation_Female_20000], ignore_index=True)
print(f"Total combined peaks (Male + Female 20kb): {len(all_peaks)}")

# Create bedtool for intersections
combined_bed = BedTool.from_dataframe(all_peaks[['Chr', 'Start', 'End']])

## 4. Combine male and female peaks

In [ ]:
# Check available gene columns
print("Male annotation columns:")
print([col for col in peak_annotation_Male_20000.columns if 'Gene' in col or 'Distance' in col])

# Extract gene columns - Gene1/Gene2 are the nearest and 2nd nearest
male_genes = set()
female_genes = set()

# Get Gene1 (nearest gene)
male_genes.update(peak_annotation_Male_20000['Gene1'].dropna().unique())
female_genes.update(peak_annotation_Female_20000['Gene1'].dropna().unique())

# Get Gene2 (2nd nearest gene)
if 'Gene2' in peak_annotation_Male_20000.columns:
    male_genes.update(peak_annotation_Male_20000['Gene2'].dropna().unique())
    female_genes.update(peak_annotation_Female_20000['Gene2'].dropna().unique())

# Remove empty strings and '.'
male_genes.discard('')
male_genes.discard('.')
female_genes.discard('')
female_genes.discard('.')

print(f"\nMale unique genes (nearest + 2nd nearest): {len(male_genes)}")
print(f"Female unique genes (nearest + 2nd nearest): {len(female_genes)}")

all_genes = male_genes | female_genes
print(f"All unique genes combined: {len(all_genes)}")
print(f"Shared genes: {len(male_genes & female_genes)}")
print(f"Male-only: {len(male_genes - female_genes)}")
print(f"Female-only: {len(female_genes - male_genes)}")

# Save gene lists
pd.DataFrame({'Gene': sorted(male_genes)}).to_csv("Male_unique_genes_20kb.txt", sep='\t', index=False, header=True)
pd.DataFrame({'Gene': sorted(female_genes)}).to_csv("Female_unique_genes_20kb.txt", sep='\t', index=False, header=True)
pd.DataFrame({'Gene': sorted(all_genes)}).to_csv("All_unique_genes_20kb.txt", sep='\t', index=False, header=True)

## 3. Extract nearest and 2nd-nearest genes

In [ ]:
# Read in bed file
bed_file = "Egr1CC_peak_FemaleEgr1_VS_FemaleWT_MACC2_window1000_YchromFiltered_window300_p05.bed"
peak_data_Female = pd.read_csv(bed_file, sep='\t', header=None)

# Add column names
peak_data_Female.columns = ['Chr', 'Start', 'End', 'Center', 'Experiment Insertions', 'Background insertions', 
                            'Reference Insertions', 'pvalue Reference', 'pvalue Background', 'Fraction Experiment', 
                            'TPH Experiment', 'Fraction background', 'TPH background', 'TPH background subtracted', 
                            'pvalue_adj Reference']

print(f"Female Egr1 peaks: {len(peak_data_Female)}")

# Annotate
peak_annotation_Female = cc.pp.annotation(peak_data_Female, reference="mm10", bedtools_path=bedtools_path)
peak_annotation_Female = cc.pp.combine_annotation(peak_data_Female, peak_annotation_Female)

# Filter to 20kb threshold
peak_annotation_Female_20000 = peak_annotation_Female[peak_annotation_Female['Distance1'].abs() <= 20000].reset_index(drop=True)
print(f"Female peaks within 20kb of gene: {len(peak_annotation_Female_20000)}")
peak_annotation_Female_20000.to_csv("Female_Egr1CC_peaks_20kbThreshhold.txt", sep="\t", index=False, header=True)

## 2. Load and annotate Female peaks

In [ ]:
# Read in bed file
bed_file = "Egr1CC_peak_MaleEgr1_VS_MaleWT_MACC2_window1000_YchromFiltered_window300_p05.bed"
peak_data_Male = pd.read_csv(bed_file, sep='\t', header=None)

# Add column names
peak_data_Male.columns = ['Chr', 'Start', 'End', 'Center', 'Experiment Insertions', 'Background insertions', 
                          'Reference Insertions', 'pvalue Reference', 'pvalue Background', 'Fraction Experiment', 
                          'TPH Experiment', 'Fraction background', 'TPH background', 'TPH background subtracted', 
                          'pvalue_adj Reference']

print(f"Male Egr1 peaks: {len(peak_data_Male)}")

# Annotate
peak_annotation_Male = cc.pp.annotation(peak_data_Male, reference="mm10", bedtools_path=bedtools_path)
peak_annotation_Male = cc.pp.combine_annotation(peak_data_Male, peak_annotation_Male)

# Filter to 20kb threshold
peak_annotation_Male_20000 = peak_annotation_Male[peak_annotation_Male['Distance1'].abs() <= 20000].reset_index(drop=True)
print(f"Male peaks within 20kb of gene: {len(peak_annotation_Male_20000)}")
peak_annotation_Male_20000.to_csv("Male_Egr1CC_peaks_20kbThreshhold.txt", sep="\t", index=False, header=True)

## 1. Load and annotate Male peaks

In [ ]:
import pandas as pd
import os
import pycallingcards as cc
from pybedtools import BedTool

os.chdir('/scratch/rmlab/rmlab_shared3/l.llaci/Egr1_paper/testing_CC_forPaper')
bedtools_path = '/ref/rmlab/software/pycallingcards/bin'

print("Working directory:", os.getcwd())

# Egr1 Peak Analysis: 20kb Filter, Gene Extraction, and Sex-Biased Comparison

Filter peaks to 20kb, extract nearest/2nd-nearest genes, then combine male/female and intersect with sex-biased peaks.